# 第5讲：视频-文本融合实践（BLIP-2模型）
## 用 BLIP-2 对数字人文短视频生成归档解说

# 本讲定位

第1讲我们让 LLaMA-3 对视频转写文本做摘要；第2讲用 CLIP 做了帧与古籍文本的跨模态对齐。
然而，数字人文归档场景中还需要一个关键能力：**直接"看懂"画面内容，并生成可检索的文字解说**。

BLIP-2 是目前最适合这一任务的轻量多模态模型：
- 输入一张视频帧，输出一句自然语言描述
- 无需文本标注，直接从图像生成解说文字
- 模型约 5 GB，Colab 免费 T4 GPU 可流畅运行

本讲将视频帧逐一送入 BLIP-2，生成每帧内容解说，最终拼合为一份结构化归档记录。

# 本讲结构

## 1  环境准备与 GPU 自检
## 2  BLIP-2 模型基础架构
## 3  加载 BLIP-2（OPT-2.7B）
## 4  检查点：确认模型可生成描述
## 5  视频帧提取与预处理
## 6  逐帧生成内容解说
## 7  拼合归档记录并保存
## 8  结果展示与质量评估
## 快速回顾

# 透明度说明

> **模型来源**：本讲使用 `Salesforce/blip2-opt-2.7b`，这是 Salesforce 团队于 2023 年发布的开源多模态模型。图像编码器（ViT）基于 MIT 许可证，但 OPT-2.7B 语言解码器受 **Meta OPT 许可证**约束（仅限非商业用途），不可用于商业产品。模型可从 Hugging Face 直接下载，无需申请或令牌。
>
> **架构简述**：BLIP-2 由三个组件构成——冻结的图像编码器（ViT）、冻结的大语言模型解码器（OPT-2.7B）、以及一个可训练的 Q-Former（查询转换器），后者充当"视觉→语言"的桥梁。
>
> **适用边界**：BLIP-2 擅长通用场景描述和简单视觉问答，但不适合高精度 OCR 或专业文物病害识别。对于低质量古籍画面的文字识别，第7讲的 LLaVA-Next 会给出更强的方案。

# 第1章  环境准备与 GPU 自检

- 安装核心依赖：`transformers`（BLIP-2 模型加载）、`torch`（深度学习框架）、`pillow`（图像处理）、`opencv-python`（视频解码）
- 运行 GPU 自检——BLIP-2 在 CPU 上每帧约 30 秒，GPU 上约 1-2 秒，差距显著

> **本地用户**：如果已在 `mllm-dh` conda 环境中运行，可注释掉 `!pip install` 行，直接执行 GPU 自检。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行，直接运行 GPU 自检。

!pip install -q transformers torch pillow opencv-python

import torch

if not torch.cuda.is_available():
    print('未检测到 GPU。Colab 用户请选择：运行时 -> 更改运行时类型 -> T4 GPU。')
    print('CPU 推理每帧约需 30 秒，GPU 约需 1-2 秒，强烈建议使用 GPU。')
else:
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 已就绪：{device_name}')
    print(f'显存总量：{total_mem:.1f} GB')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'使用设备：{DEVICE}')

# 第2章  BLIP-2 模型基础架构

## 为什么需要 BLIP-2？

将图像和文本桥接到同一个语义空间是跨模态理解的核心难题。BLIP-2 的设计思路：
**不从头训练，而是连接两个已经训练好的专家**。

| 组件 | 模型 | 作用 |
|------|------|------|
| 图像编码器 | ViT（视觉 Transformer） | 将图像转为视觉特征向量 |
| Q-Former | 可训练的查询转换器 | 从视觉特征中"提取"语言相关的信息 |
| 大语言模型 | OPT-2.7B | 将 Q-Former 的输出解码为自然语言 |

关键创新是 **Q-Former**：它用一组可学习的查询向量（Query Tokens），像"提问"一样从图像特征中提取对生成文字最有用的信息，然后喂给 LLM。

# 第2章（续） BLIP-2 的两种工作模式

| 模式 | 输入 | 输出 | 本讲使用场景 |
|------|------|------|-------------|
| **图像描述** | 图像 | 自然语言描述 | 逐帧生成画面解说 |
| **视觉问答** | 图像 + 问题 | 回答 | 可选：对特定帧提问（如"画面中是否有题跋？"） |

本讲聚焦**图像描述模式**：对每帧视频画面，自动生成英文描述，
无需额外人工标注。

> **为什么不直接用通用 LLM？** 通用 LLM（如 LLaMA-3）无法直接"看到"图像。
> BLIP-2 的 Q-Former 将视觉信号转化为 LLM 可理解的语言表征，实现了真正的多模态输入。

# 第3章  加载 BLIP-2（OPT-2.7B）

BLIP-2 有多个变体，本讲选择 **BLIP-2 + OPT-2.7B**：
- OPT-2.7B 是 Meta 发布的开源 LLM，参数量适中
- 全模型约 5 GB，加上推理中间张量，T4 的 16 GB 显存足够
- 首次加载需从 Hugging Face 下载权重，Colab 约需 2-3 分钟

In [ ]:
# ===== 第3章：加载 BLIP-2 模型与处理器 =====
# 首次运行需下载约 5 GB 权重，Colab 大约需要 2-3 分钟

from transformers import Blip2Processor, Blip2ForConditionalGeneration

MODEL_ID = 'Salesforce/blip2-opt-2.7b'

# BLIP-2 使用 Processor（非 Tokenizer），同时处理图像和文本
processor = Blip2Processor.from_pretrained(MODEL_ID)
print('处理器加载完成')

# 加载模型（自动检测并移至 GPU）
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE).eval()
print('模型加载完成')

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'参数量：{total_params:.2f} B')
print(f'模型设备：{next(model.parameters()).device}')

# 第4章  检查点：确认模型可生成描述

在提取视频帧之前，先用一张简单的测试图像验证模型加载正确、可正常推理。

这是一个**有意设置的检查点**：如果模型权重损坏或显存不足，这里会立即报错
——避免在提取了大量帧之后才发现模型不可用。

In [ ]:
# ===== 检查点：确认 BLIP-2 可生成描述 =====

from PIL import Image
import numpy as np

# 创建一张纯色测试图像（不需要真实文件）
test_img = Image.fromarray(
    np.full((224, 224, 3), fill_value=(100, 150, 200), dtype=np.uint8)
)
print(f'测试图像尺寸：{test_img.size}')

# BLIP-2 图像描述模式：仅输入图像，不传 text
inputs = processor(images=test_img, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
    )

caption = processor.decode(generated_ids[0], skip_special_tokens=True)
print('=== 检查点输出 ===')
print(f'模型对纯色图像的描述："{caption}"')
print('=' * 20)
print('BLIP-2 模型加载正常，可以继续。')

# 第5章  视频帧提取与预处理

## 帧提取策略

数字人文短视频通常时长 2-5 分钟，无需逐帧处理。本讲采用**均匀采样**策略：

| 参数 | 取值 | 说明 |
|------|------|------|
| 采样帧数 | 8 帧 | 覆盖视频关键节点，避免冗余 |
| 帧尺寸 | 224×224 | BLIP-2 ViT 编码器的标准输入尺寸 |
| 色彩空间 | RGB | PIL 图像标准格式 |

> **复用说明**：第2讲已使用 ffmpeg 提取帧，本讲改用 OpenCV，因为可以在提取同时做预处理（resize、颜色转换），减少中间文件。
>
> **提示**：如果本地已有 mp4 视频，将 `VIDEO_PATH` 改为你的文件路径即可。Colab 用户请先上传视频文件。

In [ ]:
# ===== 第5章：视频帧提取 =====

import cv2
import os

# --- 配置区 ---
# Colab 用户：上传视频后，将此处改为你的视频文件名
VIDEO_PATH = None  # 设为 None 则自动查找当前目录的 mp4/mov 文件

# 自动查找视频文件
VIDEO_LOADED = False  # 标记是否成功加载真实视频（影响后续归档记录中的 video_source 字段）
if VIDEO_PATH is None:
    import glob
    candidates = glob.glob('*.mp4') + glob.glob('*.mov') + glob.glob('*.mkv')
    if candidates:
        VIDEO_PATH = candidates[0]
        print(f'自动选择视频：{VIDEO_PATH}')
        VIDEO_LOADED = True
    else:
        print('⚠️  未找到视频文件，将使用测试图像占位。')
        print('⚠️  如果你已上传视频，请检查文件扩展名（.mp4 / .mov / .mkv）或将 VIDEO_PATH 设为完整路径。')
else:
    VIDEO_LOADED = True

NUM_FRAMES = 8

def extract_frames(video_path, num_frames=NUM_FRAMES):
    """从视频中均匀采样 num_frames 帧，返回 PIL Image 列表。"""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None, 0

    # 均匀采样帧索引
    indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        # BGR -> RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(frame_rgb).resize((224, 224))
        frames.append(img)

    cap.release()
    return frames, total_frames

if VIDEO_LOADED and VIDEO_PATH and os.path.exists(VIDEO_PATH):
    frames, total = extract_frames(VIDEO_PATH)
    if frames is None or len(frames) == 0:
        print(f'⚠️  警告：视频文件可能损坏（总帧数={total}），回退至测试图像。')
        print(f'⚠️  后续生成的解说将基于纯色占位图，并非真实视频内容！')
        VIDEO_LOADED = False
        frames = [Image.fromarray(
            np.full((224, 224, 3), fill_value=(c*30, 100+c*15, 200-c*20), dtype=np.uint8)
        ) for c in range(NUM_FRAMES)]
    elif len(frames) < NUM_FRAMES:
        print(f'⚠️  警告：仅提取到 {len(frames)}/{NUM_FRAMES} 帧（预期 {NUM_FRAMES} 帧），视频可能较短或读取中断。')
        print(f'✅ 真实视频已加载：{VIDEO_PATH}')
        print(f'视频总帧数：{total}')
        print(f'已提取 {len(frames)} 帧（均匀采样）')
    else:
        print(f'✅ 真实视频已加载：{VIDEO_PATH}')
        print(f'视频总帧数：{total}')
        print(f'已提取 {len(frames)} 帧（均匀采样）')
else:
    # 使用测试图像占位
    print('⚠️  当前使用测试图像占位——后续生成的解说并非真实视频内容！')
    frames = [Image.fromarray(
        np.full((224, 224, 3), fill_value=(c*30, 100+c*15, 200-c*20), dtype=np.uint8)
    ) for c in range(NUM_FRAMES)]
    print(f'已生成 {len(frames)} 帧测试图像（占位）')

print(f'每帧尺寸：{frames[0].size}')

# 第6章  逐帧生成内容解说

## BLIP-2 的图像描述流程

对每一帧，BLIP-2 执行以下步骤：
1. ViT 编码器将 224×224 图像转为视觉特征图
2. Q-Former 用 32 个查询向量从特征图中提取"语言化"表示
3. OPT-2.7B 将表示解码为自然语言描述

BLIP-2 的 OPT-2.7B 解码器基于英文语料训练，输出为英文描述。中文归档需求可通过第 1 讲的 LLaMA-3 做翻译后处理，或等待第 7 讲 LLaVA-Next 的原生多语言支持。本讲聚焦英文描述生成与归档。

In [ ]:
# ===== 第6章：逐帧生成解说 =====

captions = []

print(f'正在为 {len(frames)} 帧生成解说...')
for i, frame in enumerate(frames):
    # 使用 BLIP-2 图像描述模式生成英文描述
    prompt = 'Question: Describe this image in one sentence. Answer:'
    inputs = processor(images=frame, text=prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        )

    # 仅解码新生成的 token（排除输入 prompt）
    caption = processor.decode(
        generated_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    captions.append(caption)
    print(f'  帧 {i+1}/{len(frames)}：{caption[:60]}...')

print(f'\n全部 {len(captions)} 帧解说生成完成。')

# 第7章  拼合归档记录并保存

## 从单帧描述到完整归档

单帧描述是碎片化的。数字人文归档需要将这些片段整合为一个结构化的元数据记录。

本讲将帧级解说拼合为以下归档格式：
- **视频标识**：文件名 + 处理时间戳
- **帧列表**：帧序号 + 对应解说文本
- **视频级摘要**：拼接全部帧描述形成完整解说

这一格式可直接用于后续的文本检索（第6讲 Whisper 转写衔接）、元数据入库等流程。

In [ ]:
# ===== 第7章：拼合归档记录 =====

import json
from datetime import datetime

# 构建归档记录
# 注：video_summary 为所有帧描述的简单拼接（concatenated captions），
# 并非由 LLM 生成的独立摘要段落。如需深度摘要，可接入第1讲的 LLaMA-3。
archive_record = {
    'video_source': VIDEO_PATH if VIDEO_LOADED else 'test_placeholder',
    'processed_at': datetime.now().isoformat(),
    'model': MODEL_ID,
    'num_frames': len(frames),
    'frames': [
        {
            'frame_index': i,
            'caption': captions[i],
        }
        for i in range(len(frames))
    ],
    'video_summary': ' '.join(captions),
}

# 保存为 JSON 归档文件
OUTPUT_PATH = 'L05_archive_output.json'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(archive_record, f, ensure_ascii=False, indent=2)

print(f'归档记录已保存至：{OUTPUT_PATH}')
print(f'视频级摘要（帧描述拼接）：')
print(archive_record['video_summary'][:200] + ('...' if len(archive_record['video_summary']) > 200 else ''))
print()
print('💡 提示：Colab 运行时重置后文件会丢失，请及时下载 JSON 或挂载 Google Drive 保存。')

# 第8章  结果展示与质量评估

## BLIP-2 解说质量的参考维度

将帧描述与实际画面做人工对比，评估以下三个维度：

| 维度 | 检查要点 |
|------|----------|
| **准确性** | 描述是否与画面内容吻合？是否出现幻觉（描述不存在的物体）？ |
| **信息量** | 是否捕捉到数字人文相关的关键视觉元素（文字、文物、场景）？ |
| **语言质量** | 英文是否通顺自然？是否存在重复或截断？ |

> **已知局限**：BLIP-2 的 OPT-2.7B 解码器仅支持英文输出，无法生成中文描述。如需中文归档，可在本讲的英文描述基础上接入第 1 讲的 LLaMA-3 做翻译，或使用第 7 讲的 LLaVA-Next 实现原生多语言视觉问答。

In [ ]:
# ===== 第8章：逐帧解说展示 =====

print('===== BLIP-2 视频帧解说结果 =====')
print(f'处理时间：{archive_record["processed_at"]}')
print(f'帧数量：{archive_record["num_frames"]}')
print()
for frame_info in archive_record['frames']:
    i = frame_info['frame_index']
    caption = frame_info['caption']
    print(f'--- 帧 {i+1} ---')
    print(f'  解说：{caption}')
    print()

print()
print('提示：可人工抽查前 3 帧描述是否与实际画面匹配，参考上方质量评估维度进行判断。')

# 快速回顾（Quick Check）

**本讲完成了以下工作：**

- **BLIP-2 加载**：使用 `Salesforce/blip2-opt-2.7b`，约 5 GB 权重，T4 GPU 可流畅运行
- **帧提取**：从视频中均匀采样 8 帧，统一缩放至 224x224
- **逐帧解说**：为每帧生成英文描述性文字
- **归档输出**：拼合帧级解说为结构化 JSON 记录，可直接用于后续检索系统

> **和前后各讲的连贯性**：L01-L02 分别处理文本摘要和帧-文本对齐；L05 让模型直接"看图说话"；
> L06 将引入 Whisper 语音转写，实现视频音频→文本的联动归档。